# Chapter 13: Project - Sentiment Analysis Dashboard

In this chapter, we develop a **sentiment analysis dashboard** -- an end-to-end NLP application that lets users upload text data, analyze sentiment using trained models, and visualize the results through an interactive web interface.

**Sentiment analysis** (also called opinion mining) is an NLP technique used to determine the emotional tone behind a body of text. It is widely used in business to understand customer opinions, monitor brand reputation, and gain insights into market trends.

This project covers five stages:
1. **Project Introduction and Design** -- architecture, requirements, project structure
2. **Data Collection and Preprocessing** -- loading data, cleaning text, handling imbalanced classes
3. **Building and Training Models** -- Logistic Regression, LSTM, hyperparameter tuning
4. **Developing the Dashboard Interface** -- Flask web app, HTML templates, Plotly visualizations, file uploads
5. **Evaluating and Deploying** -- accuracy metrics, user feedback, response time, Heroku deployment, Slack integration

## Install Required Packages

Run the cell below to install all dependencies needed for this project.

In [ ]:
!pip install pandas scikit-learn nltk imbalanced-learn tensorflow flask plotly requests

In [ ]:
# Download required NLTK data
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

---
## 13.1 Project Introduction and Design

### 13.1.1 Project Overview

The objective is to create a sentiment analysis dashboard that provides users with a comprehensive tool to analyze textual data. The dashboard will enable users to:

- **Upload** text data for sentiment analysis
- **Visualize** the sentiment distribution and trends over time
- **Gain insights** into the overall sentiment of the text data

### 13.1.2 Design Considerations

Designing a sentiment analysis dashboard involves several considerations:

1. **Data Sources** -- social media posts, customer reviews, survey responses, or any textual data
2. **Sentiment Analysis Model** -- classify text as positive, negative, or neutral
3. **User Interface** -- intuitive and interactive web-based dashboard
4. **Visualization Tools** -- clear and informative charts
5. **Scalability and Performance** -- handle large volumes of text with real-time results

### 13.1.3 System Architecture

The system architecture consists of five components:

| Component | Description |
|---|---|
| **Frontend Interface** | Web-based UI where users interact with the dashboard |
| **Backend Server** | Handles requests, processes text, interfaces with the model |
| **Sentiment Analysis Model** | NLP model that classifies text as positive/negative/neutral |
| **Database** | Stores uploaded data, results, and user preferences |
| **Visualization Engine** | Generates interactive charts from analysis results |

### 13.1.4 Implementation Plan

1. Define requirements
2. Set up project structure
3. Collect and preprocess data
4. Build and train the sentiment analysis model
5. Develop the dashboard interface
6. Integrate components and test
7. Deploy the dashboard

### 13.1.5 Project Structure

```
sentiment_analysis_dashboard/
├── data/
│   ├── raw_data/           # Raw text data uploaded by users
│   ├── processed_data/     # Preprocessed data ready for analysis
├── models/
│   ├── sentiment_model.h5  # Trained deep learning model
│   └── tokenizer.pickle    # Fitted tokenizer
├── scripts/
│   ├── data_preprocessing.py
│   ├── sentiment_analysis.py
│   └── dashboard.py
├── templates/
│   └── index.html
├── static/
│   ├── css/
│   └── js/
├── app.py                  # Main application entry point
└── requirements.txt
```

### 13.1.6 Defining Requirements

The dashboard should be able to:

1. Upload text data from various sources
2. Preprocess text data for sentiment analysis
3. Analyze sentiment using a trained model
4. Visualize sentiment distribution and trends over time
5. Allow users to explore results through an intuitive interface
6. Store user preferences and interaction history

---
## 13.2 Data Collection and Preprocessing

Data collection and preprocessing are crucial steps. The quality of the data and how it is processed directly impact the performance and reliability of the sentiment analysis model.

### 13.2.1 Collecting Data

To perform sentiment analysis we need a labeled dataset (positive, negative, or neutral). Common sources include:

- **Public Datasets** -- IMDB Movie Reviews, Twitter Sentiment140, Yelp Reviews
- **User-Generated Data** -- customer reviews, social media posts, survey responses

For this project, we use the **IMDB Movie Reviews dataset** (50,000 reviews labeled as positive or negative). You can download it from [Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews).

Below we demonstrate loading and splitting the dataset. For demonstration purposes, we create a small synthetic dataset so the notebook runs without the full download.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# --- Option A: Load the real IMDB dataset (uncomment if you have the CSV) ---
# data = pd.read_csv('data/raw_data/IMDB_Dataset.csv')

# --- Option B: Create a small synthetic dataset for demonstration ---
reviews = [
    "I absolutely loved this movie, it was fantastic!",
    "Terrible film, a complete waste of time.",
    "Great acting and a wonderful storyline.",
    "I hated every minute of this boring movie.",
    "An amazing experience, highly recommend it!",
    "The worst movie I have ever seen.",
    "Beautiful cinematography and a touching story.",
    "Awful plot with bad acting throughout.",
    "One of the best films of the year.",
    "Dull and predictable, not worth watching.",
    "Superb direction and an engaging narrative.",
    "A disappointing sequel that fails to deliver.",
    "Heartwarming and uplifting, a true masterpiece.",
    "Poorly written script with no redeeming qualities.",
    "Incredible performances by the entire cast.",
    "Boring and overlong, I almost fell asleep.",
    "A thrilling adventure from start to finish.",
    "The dialogue was cringe-worthy and unnatural.",
    "I was moved to tears, absolutely brilliant.",
    "A forgettable movie with zero originality.",
]
sentiments = ['positive', 'negative'] * 10  # alternating

data = pd.DataFrame({'review': reviews, 'sentiment': sentiments})
print(f"Dataset shape: {data.shape}")
print(data.head())

In [ ]:
# Split into training and test sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

print(f"Training set size: {len(train_data)}")
print(f"Test set size:     {len(test_data)}")
print(f"\nSentiment distribution (train):")
print(train_data['sentiment'].value_counts())

### 13.2.2 Preprocessing Data

The preprocessing pipeline includes:
- **Text normalization** -- convert to lowercase, remove punctuation
- **Tokenization** -- split text into individual words/tokens
- **Stop word removal** -- remove common words that don't carry sentiment
- **Lemmatization** -- reduce words to their base form
- **Vectorization** -- convert text to numerical features (TF-IDF)

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import string

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Normalize, tokenize, remove stop words, and lemmatize text."""
    # Convert to lowercase
    text = text.lower()
    # Tokenize
    tokens = nltk.word_tokenize(text)
    # Remove punctuation and stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in string.punctuation and word not in stop_words]
    # Lemmatize tokens
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

# Demonstrate preprocessing on a single example
sample = "I absolutely loved this movie, it was fantastic!"
print(f"Original:     {sample}")
print(f"Preprocessed: {preprocess_text(sample)}")

In [ ]:
# Apply preprocessing to training and test sets
train_data = train_data.copy()
test_data = test_data.copy()
train_data['review'] = train_data['review'].apply(preprocess_text)
test_data['review'] = test_data['review'].apply(preprocess_text)

print("Preprocessed training examples:")
print(train_data[['review', 'sentiment']].head())

In [ ]:
# Vectorize the preprocessed text using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_data['review']).toarray()
X_test = vectorizer.transform(test_data['review']).toarray()

# Encode sentiment labels
y_train = train_data['sentiment'].apply(lambda x: 1 if x == 'positive' else 0).values
y_test = test_data['sentiment'].apply(lambda x: 1 if x == 'positive' else 0).values

print(f"TF-IDF matrix shape (train): {X_train.shape}")
print(f"TF-IDF matrix shape (test):  {X_test.shape}")
print(f"\nTop 10 features: {vectorizer.get_feature_names_out()[:10]}")

### 13.2.3 Handling Imbalanced Data with SMOTE

In real-world datasets the distribution of sentiment labels is often imbalanced. The **Synthetic Minority Over-sampling Technique (SMOTE)** generates synthetic samples for the minority class to balance the dataset.

In [ ]:
from imblearn.over_sampling import SMOTE
import numpy as np

print(f"Original class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")

# Apply SMOTE to balance the dataset
# Note: with our small balanced demo dataset SMOTE has no effect,
# but on a real imbalanced dataset this step is critical.
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Balanced class distribution: {dict(zip(*np.unique(y_train_balanced, return_counts=True)))}")
print(f"Balanced training set shape: {X_train_balanced.shape}")

---
## 13.3 Building and Training Sentiment Analysis Models

We explore two approaches:
1. **Traditional ML** -- Logistic Regression (fast, interpretable)
2. **Deep Learning** -- LSTM (captures sequential patterns, higher accuracy on large datasets)

### 13.3.1 Choosing the Right Model

| Approach | Models | Pros | Cons |
|---|---|---|---|
| Traditional ML | Logistic Regression, SVM, Naive Bayes | Fast, interpretable, less data needed | May miss complex patterns |
| Deep Learning | RNN, LSTM, BERT | Captures sequential/contextual patterns | More compute, more data needed |

### 13.3.2 Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Train a Logistic Regression model on the balanced data
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_balanced, y_train_balanced)

# Evaluate on the test set
y_pred_lr = lr_model.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print(f"Logistic Regression Accuracy: {accuracy_lr:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

### 13.3.3 LSTM Deep Learning Model

For the LSTM model, we tokenize the text into integer sequences and pad them to a fixed length before feeding them into an embedding + LSTM network.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Prepare text sequences for LSTM
X_train_text = train_data['review'].values
X_test_text = test_data['review'].values

# Tokenize and pad sequences
keras_tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
keras_tokenizer.fit_on_texts(X_train_text)

X_train_sequences = keras_tokenizer.texts_to_sequences(X_train_text)
X_test_sequences = keras_tokenizer.texts_to_sequences(X_test_text)

max_length = 200
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_length, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_length, padding='post', truncating='post')

print(f"Padded training shape: {X_train_padded.shape}")
print(f"Padded test shape:     {X_test_padded.shape}")
print(f"Vocabulary size:       {len(keras_tokenizer.word_index)}")

In [ ]:
# Build the LSTM model
embedding_dim = 100

lstm_model = Sequential([
    Embedding(input_dim=5000, output_dim=embedding_dim, input_length=max_length),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

In [ ]:
# Train the LSTM model
# Note: with our small demo dataset results will be limited;
# on the full IMDB dataset (50k reviews), expect ~85-90% accuracy.
history = lstm_model.fit(
    X_train_padded, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate on test set
y_pred_prob = lstm_model.predict(X_test_padded)
y_pred_lstm = (y_pred_prob > 0.5).astype(int).flatten()

accuracy_lstm = accuracy_score(y_test, y_pred_lstm)
print(f"\nLSTM Accuracy: {accuracy_lstm:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lstm, target_names=['Negative', 'Positive']))

### 13.3.4 Hyperparameter Tuning with GridSearchCV

Hyperparameter tuning optimizes model performance by searching over a grid of parameter combinations.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grid for Logistic Regression
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear']
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1
)

# Perform hyperparameter tuning
grid_search.fit(X_train_balanced, y_train_balanced)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score:   {grid_search.best_score_:.4f}")

# Use the best model
best_model = grid_search.best_estimator_

### 13.3.5 Evaluating Model Performance

We evaluate using accuracy, precision, recall, F1-score, and a confusion matrix visualization.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Predict with the best Logistic Regression model
y_pred_best = best_model.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)

print(f"Best LR Model Accuracy: {accuracy_best:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Negative', 'Positive']))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred_best, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Logistic Regression - Confusion Matrix')
plt.show()

---
## 13.4 Developing the Dashboard Interface

We use **Flask** (a lightweight Python web framework) for the backend, **Bootstrap** for responsive styling, and **Plotly** for interactive visualizations.

### 13.4.1 Setting Up Flask

The Flask application loads both trained models and exposes routes for:
- `/` -- home page with a text input form
- `/analyze` -- performs sentiment analysis on submitted text
- `/visualize` -- shows sentiment distribution charts
- `/upload` -- allows CSV file upload

Below is the complete `app.py` for the dashboard.

In [ ]:
# ============================================================
# app.py -- Sentiment Analysis Dashboard (Flask application)
# ============================================================
# NOTE: This cell shows the complete app.py source code.
#       To run the dashboard, save this as app.py and execute:
#       python app.py
# ============================================================

app_source = r'''
from flask import Flask, request, render_template, jsonify
import pickle
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import plotly.express as px
import pandas as pd
import os
from werkzeug.utils import secure_filename

app = Flask(__name__)

# ----- Load models and tokenizer -----
lstm_model = load_model("models/lstm_model.h5")
with open("models/tokenizer.pickle", "rb") as f:
    tokenizer = pickle.load(f)
with open("models/best_logistic_regression_model.pickle", "rb") as f:
    logistic_regression_model = pickle.load(f)

# ----- File upload config -----
UPLOAD_FOLDER = "data/uploads"
ALLOWED_EXTENSIONS = {"csv"}
app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER

def allowed_file(filename):
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS

def preprocess_text(text):
    max_length = 200
    sequences = tokenizer.texts_to_sequences([text])
    padded_sequences = pad_sequences(sequences, maxlen=max_length,
                                     padding="post", truncating="post")
    return padded_sequences

# ----- Routes -----
@app.route("/")
def home():
    return render_template("index.html")

@app.route("/analyze", methods=["POST"])
def analyze():
    text = request.form["text"]
    model_type = request.form["model_type"]
    preprocessed_text = preprocess_text(text)
    if model_type == "lstm":
        prediction_prob = lstm_model.predict(preprocessed_text)
        prediction = (prediction_prob > 0.5).astype(int)
    else:
        prediction = logistic_regression_model.predict(preprocessed_text)
    sentiment = "Positive" if prediction[0] == 1 else "Negative"
    return jsonify({"sentiment": sentiment})

@app.route("/visualize", methods=["GET"])
def visualize():
    data = {
        "Text": ["I love this!", "I hate this!", "This is amazing!", "This is terrible!"],
        "Sentiment": ["Positive", "Negative", "Positive", "Negative"]
    }
    df = pd.DataFrame(data)
    fig = px.histogram(df, x="Sentiment", title="Sentiment Distribution")
    graph_html = fig.to_html(full_html=False)
    return render_template("visualize.html", graph_html=graph_html)

@app.route("/upload", methods=["GET", "POST"])
def upload_file():
    if request.method == "POST":
        if "file" not in request.files:
            return jsonify({"message": "No file part"})
        file = request.files["file"]
        if file.filename == "":
            return jsonify({"message": "No selected file"})
        if file and allowed_file(file.filename):
            filename = secure_filename(file.filename)
            file.save(os.path.join(app.config["UPLOAD_FOLDER"], filename))
            return jsonify({"message": "File uploaded successfully"})
    return render_template("upload.html")

if __name__ == "__main__":
    app.run(debug=True)
'''

print(app_source)

### 13.4.2 HTML Templates

The main template (`templates/index.html`) provides a form for entering text, selecting a model, and submitting for analysis. It uses **Bootstrap** for responsive styling and **jQuery** for AJAX form submission.

In [ ]:
# templates/index.html
index_html = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sentiment Analysis Dashboard</title>
    <link rel="stylesheet"
          href="https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css">
</head>
<body>
    <div class="container">
        <h1 class="mt-5">Sentiment Analysis Dashboard</h1>
        <form id="analysis-form" class="mt-4">
            <div class="form-group">
                <label for="text">Enter Text</label>
                <textarea class="form-control" id="text" name="text"
                          rows="3" required></textarea>
            </div>
            <div class="form-group">
                <label for="model_type">Select Model</label>
                <select class="form-control" id="model_type" name="model_type" required>
                    <option value="logistic_regression">Logistic Regression</option>
                    <option value="lstm">LSTM</option>
                </select>
            </div>
            <button type="submit" class="btn btn-primary">Analyze Sentiment</button>
        </form>
        <div id="result" class="mt-4"></div>
    </div>

    <script src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
    <script>
    $(document).ready(function() {
        $("#analysis-form").on("submit", function(event) {
            event.preventDefault();
            const formData = $(this).serialize();
            $.post("/analyze", formData, function(data) {
                $("#result").html(`<h3>Sentiment: ${data.sentiment}</h3>`);
            });
        });
    });
    </script>
</body>
</html>
'''

print(index_html)

### 13.4.3 Data Visualization with Plotly

Plotly creates interactive, browser-based charts. Below we demonstrate generating a sentiment distribution histogram that would be served via the `/visualize` route.

In [ ]:
import plotly.express as px
import pandas as pd

# Example sentiment results
viz_data = {
    'Text': [
        'I love this product!', 'Terrible experience.', 'Absolutely wonderful!',
        'Worst purchase ever.', 'Great quality and fast shipping.',
        'Broke after one day.', 'Highly recommend to everyone!',
        'Complete waste of money.'
    ],
    'Sentiment': ['Positive', 'Negative', 'Positive', 'Negative',
                  'Positive', 'Negative', 'Positive', 'Negative']
}
df_viz = pd.DataFrame(viz_data)

# Sentiment distribution histogram
fig = px.histogram(df_viz, x='Sentiment', title='Sentiment Distribution',
                   color='Sentiment',
                   color_discrete_map={'Positive': '#2ecc71', 'Negative': '#e74c3c'})
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Visualize sentiment as a pie chart
sentiment_counts = df_viz['Sentiment'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Count']

fig_pie = px.pie(sentiment_counts, values='Count', names='Sentiment',
                 title='Sentiment Proportion',
                 color='Sentiment',
                 color_discrete_map={'Positive': '#2ecc71', 'Negative': '#e74c3c'})
fig_pie.show()

### 13.4.4 File Upload Functionality

The `/upload` route accepts CSV files. The `upload.html` template provides a simple form, and the backend uses `werkzeug.utils.secure_filename` for safe file handling.

In [ ]:
# templates/upload.html
upload_html = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Upload Text Data</title>
    <link rel="stylesheet"
          href="https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css">
</head>
<body>
    <div class="container">
        <h1 class="mt-5">Upload Text Data for Sentiment Analysis</h1>
        <form action="/upload" method="post" enctype="multipart/form-data" class="mt-4">
            <div class="form-group">
                <label for="file">Choose CSV File</label>
                <input type="file" class="form-control-file"
                       id="file" name="file" required>
            </div>
            <button type="submit" class="btn btn-primary">Upload</button>
        </form>
    </div>
</body>
</html>
'''

print(upload_html)

### 13.4.5 Visualization Template

The `/visualize` route renders a Plotly chart inside the `visualize.html` template using the `| safe` Jinja2 filter to inject raw HTML.

In [ ]:
# templates/visualize.html
visualize_html = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sentiment Analysis Visualization</title>
    <link rel="stylesheet"
          href="https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css">
</head>
<body>
    <div class="container">
        <h1 class="mt-5">Sentiment Analysis Visualization</h1>
        <div id="graph-container">
            {{ graph_html | safe }}
        </div>
    </div>
</body>
</html>
'''

print(visualize_html)

---
## 13.5 Evaluating and Deploying the Dashboard

Evaluation ensures the dashboard meets user expectations. Deployment makes it accessible to users.

### 13.5.1 Accuracy Metrics

We compare both models side by side using precision, recall, F1-score, and confusion matrices.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

# --- Logistic Regression evaluation ---
y_pred_lr_final = best_model.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr_final)

print("=" * 50)
print("LOGISTIC REGRESSION")
print("=" * 50)
print(f"Accuracy: {acc_lr:.4f}")
print(classification_report(y_test, y_pred_lr_final, target_names=['Negative', 'Positive']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm_lr = confusion_matrix(y_test, y_pred_lr_final, labels=[0, 1])
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Negative', 'Positive'])
disp_lr.plot(cmap=plt.cm.Blues, ax=axes[0])
axes[0].set_title('Logistic Regression')

# --- LSTM evaluation ---
y_pred_lstm_final = (lstm_model.predict(X_test_padded) > 0.5).astype(int).flatten()
acc_lstm = accuracy_score(y_test, y_pred_lstm_final)

print("=" * 50)
print("LSTM")
print("=" * 50)
print(f"Accuracy: {acc_lstm:.4f}")
print(classification_report(y_test, y_pred_lstm_final, target_names=['Negative', 'Positive']))

cm_lstm = confusion_matrix(y_test, y_pred_lstm_final, labels=[0, 1])
disp_lstm = ConfusionMatrixDisplay(confusion_matrix=cm_lstm, display_labels=['Negative', 'Positive'])
disp_lstm.plot(cmap=plt.cm.Oranges, ax=axes[1])
axes[1].set_title('LSTM')

plt.tight_layout()
plt.show()

### 13.5.2 User Feedback Collection

User feedback is essential for assessing usability and satisfaction. We add a `/feedback` route to the Flask app and an HTML form for collecting ratings and comments.

In [ ]:
# Feedback route to add to app.py
feedback_code = '''
feedback_data = []

@app.route("/feedback", methods=["POST"])
def feedback():
    user_feedback = request.json
    feedback_data.append(user_feedback)
    return jsonify({"message": "Thank you for your feedback!"})
'''

# Feedback form HTML snippet (add to index.html or a dedicated page)
feedback_html = '''
<form id="feedback-form" class="mt-4">
    <div class="form-group">
        <label for="rating">Rate your experience:</label>
        <select class="form-control" id="rating" name="rating">
            <option value="1">1 - Poor</option>
            <option value="2">2 - Fair</option>
            <option value="3">3 - Good</option>
            <option value="4">4 - Very Good</option>
            <option value="5">5 - Excellent</option>
        </select>
    </div>
    <div class="form-group">
        <label for="comments">Comments:</label>
        <textarea class="form-control" id="comments" name="comments" rows="3"></textarea>
    </div>
    <button type="submit" class="btn btn-primary">Submit Feedback</button>
</form>

<script>
$("#feedback-form").on("submit", function(event) {
    event.preventDefault();
    const formData = $(this).serializeArray().reduce((obj, item) => {
        obj[item.name] = item.value;
        return obj;
    }, {});
    $.ajax({
        url: "/feedback",
        type: "POST",
        contentType: "application/json",
        data: JSON.stringify(formData),
        success: function(data) {
            $("#feedback-result").html(`<h4>${data.message}</h4>`);
        }
    });
});
</script>
'''

print("Feedback route code:")
print(feedback_code)
print("\nFeedback HTML form:")
print(feedback_html)

### 13.5.3 Measuring Response Time

Response time is critical for user experience. We measure how long each prediction takes to ensure the dashboard is responsive.

In [ ]:
import time
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Measure response time for the Logistic Regression model
sample_text = "This is a great product!"
preprocessed = preprocess_text(sample_text)
sample_vec = vectorizer.transform([preprocessed]).toarray()

start = time.time()
_ = best_model.predict(sample_vec)
lr_time = time.time() - start
print(f"Logistic Regression prediction time: {lr_time:.6f} seconds")

# Measure response time for the LSTM model
sample_seq = keras_tokenizer.texts_to_sequences([preprocessed])
sample_padded = pad_sequences(sample_seq, maxlen=max_length, padding='post', truncating='post')

start = time.time()
_ = lstm_model.predict(sample_padded, verbose=0)
lstm_time = time.time() - start
print(f"LSTM prediction time:                {lstm_time:.6f} seconds")

if lr_time > 0:
    print(f"\nLR is {lstm_time / lr_time:.1f}x faster than LSTM for single predictions.")

In [ ]:
# Alternatively, measure response time by calling the Flask app endpoint
# (requires the Flask server to be running separately)

response_time_code = '''
import time
import requests

def measure_response_time(endpoint, data):
    """Send a POST request and measure round-trip time."""
    start_time = time.time()
    response = requests.post(endpoint, data=data)
    end_time = time.time()
    return end_time - start_time

# Example usage (with the Flask server running on localhost:5000)
data = {"text": "This is a great product!", "model_type": "logistic_regression"}
response_time = measure_response_time("http://localhost:5000/analyze", data)
print(f"Sentiment Analysis Response Time: {response_time:.4f} seconds")
'''

print(response_time_code)

### 13.5.4 Deploying on Heroku

To deploy the dashboard as a web application, we use **Heroku**. The key steps are:

1. **Install Heroku CLI** and log in: `heroku login`
2. **Create a Heroku app**: `heroku create your-app-name`
3. **Create deployment files**:
   - `Procfile` -- tells Heroku how to run the app
   - `requirements.txt` -- lists Python dependencies
4. **Push to Heroku**:
   ```bash
   git init
   git add .
   git commit -m "Initial commit"
   git push heroku master
   ```
5. **Open the app**: `heroku open`

In [ ]:
# Procfile content
procfile = "web: python app.py"
print(f"Procfile:\n{procfile}")

# requirements.txt content
requirements = """Flask
requests
pandas
scikit-learn
tensorflow
plotly
imbalanced-learn
nltk
gunicorn"""
print(f"\nrequirements.txt:\n{requirements}")

### 13.5.5 Integrating with Slack

For real-time sentiment analysis updates, we can integrate the dashboard with **Slack** using incoming webhooks.

Steps:
1. Create a Slack App on the [Slack API](https://api.slack.com/apps)
2. Enable incoming webhooks
3. Add a `/slack` route to the Flask app

In [ ]:
# Slack integration route (add to app.py)
slack_code = '''
SLACK_WEBHOOK_URL = "your_slack_webhook_url"

@app.route("/slack", methods=["POST"])
def slack():
    data = request.json
    text = data["text"]

    # Perform sentiment analysis
    preprocessed_text = preprocess_text(text)
    prediction = logistic_regression_model.predict(preprocessed_text)
    sentiment = "Positive" if prediction[0] == 1 else "Negative"

    # Send result back to Slack
    response = {
        "response_type": "in_channel",
        "text": f"Sentiment: {sentiment}"
    }
    return jsonify(response)
'''

print(slack_code)

---
## Chapter Summary

In this chapter we built a complete **Sentiment Analysis Dashboard** covering five stages:

| Stage | What We Did |
|---|---|
| **Project Design** | Defined requirements, system architecture, and project structure |
| **Data Collection and Preprocessing** | Loaded IMDB data; applied normalization, tokenization, stop word removal, lemmatization, TF-IDF vectorization; balanced classes with SMOTE |
| **Model Building and Training** | Trained Logistic Regression and LSTM models; tuned hyperparameters with GridSearchCV |
| **Dashboard Interface** | Built a Flask web app with Bootstrap UI, Plotly visualizations, and CSV file upload |
| **Evaluation and Deployment** | Compared model accuracy with confusion matrices; measured response time; deployed to Heroku; integrated with Slack |

The skills and techniques from this project can be extended to build more sophisticated analytical dashboards with additional models, real-time data streams, and richer visualizations.